# 02 · Keyword dataset: download + record your own

1. **Google Speech Commands v0.02** (CC-BY 4.0). The 2.3 GB archive is streamed
   and only this project's words are kept (about 600 MB): ten demo command words,
   a slice of the rest as *unknown*, and the background-noise recordings.
2. **Your own voice**, Bangla and English, through this laptop's microphone, for
   the device's real vocabulary (`firmware/stm32h743/tools/keywords.csv`).
   No public Bangla keyword dataset exists for these words; this is how it is made.

Recordings stay on this computer (`data/`, git-ignored). Record a person only with
their consent.

In [ ]:
import sys, pathlib
LAB = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(LAB))
from va_lab import config
config.ensure_dirs()
print("lab folder:", LAB)

In [ ]:
from va_lab import kws_data
kws_data.download_speech_commands()   # resumable: re-running skips if complete

In [ ]:
import collections, pathlib
counts = collections.Counter(p.parent.name for p in config.SPEECH_COMMANDS_DIR.glob("*/*.wav"))
print(dict(sorted(counts.items())))

In [ ]:
# Listen to a few examples
import random, sounddevice as sd
from va_lab import audio_io
for w in ["stop", "go", "left"]:
    f = random.choice(sorted((config.SPEECH_COMMANDS_DIR / w).glob("*.wav")))
    print(w, f.name); sd.play(audio_io.load_wav(f), 16000); sd.wait()

## Record your own keywords

Set `SPEAKER` to an anonymous code (S01, S02, …). Each run asks for every
keyword `REPS` times in random order, then ordinary words for the *unknown*
class, then optional background noise. Say each word after **now**.

In [ ]:
import random, time
from va_lab import audio_io
keywords = config.load_keywords()
for k in keywords: print(f"{k['label']:14s} {k['say']}  ({k['romanised']})")

In [ ]:
SPEAKER = "S01"   # anonymous code
REPS = 5          # 10+ per speaker is better; 30+ speakers is the goal

def next_index(label):
    d = config.MY_KEYWORDS_DIR / label
    return len(list(d.glob(f"{label}.{SPEAKER}_*.wav"))) + 1

def take(label, prompt):
    while True:
        print(f"\nSay:  {prompt}"); time.sleep(0.5); print("  ... now")
        clip = audio_io.loudest_second(audio_io.record(config.KWS_LISTEN_S))
        level = audio_io.dbfs(clip)
        if level < -40: print(f"  too quiet ({level:.0f} dBFS), again"); continue
        if level > -0.5: print("  clipped, again a little further away"); continue
        path = config.MY_KEYWORDS_DIR / label / f"{label}.{SPEAKER}_{next_index(label)}.wav"
        audio_io.save_wav(path, clip); print("  saved", path.name, f"({level:.0f} dBFS)"); return

input("Quiet room, laptop at arm's length. Press Enter to start.")
for r in range(REPS):
    for k in random.sample(keywords, len(keywords)):
        take(k["label"], f"{k['say']}   ({k['romanised']})" if k["romanised"] else k["say"])

In [ ]:
# Ordinary words for the "_unknown" class (near-misses matter most)
FILLER = ["আম", "পানি", "কলম", "বাজার", "চা", "বই", "মোবাইল", "ভাত", "জামা", "চাবি", "থাকো", "আবার", "জানালা",
          "apple", "water", "pen", "phone", "hello", "okay", "thanks", "listen"]
for w in random.sample(FILLER, 10):
    take("_unknown", w)

In [ ]:
# Background noise: nobody speaking; fan, street, kitchen... (seconds)
SECONDS = 60
input(f"Leave the laptop somewhere noisy and press Enter; recording {SECONDS} s")
x = audio_io.record(SECONDS)
for i in range(0, len(x) - 16000 + 1, 16000):
    audio_io.save_wav(config.MY_KEYWORDS_DIR / "_noise" / f"_noise.{SPEAKER}_{i // 16000:03d}.wav", x[i:i + 16000])
print("saved", len(x) // 16000, "noise clips")

In [ ]:
counts = collections.Counter(p.parent.name for p in config.MY_KEYWORDS_DIR.glob("*/*.wav"))
print(dict(sorted(counts.items())))